# Correction 04 : choisir k et normaliser avec KNN

Cet exercice montre que KNN est une méthode mathématique, mais que certains choix comme `k` relèvent d'une recette pratique à valider expérimentalement.

## 1. Charger Iris et faire le split

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

## 2. Tester plusieurs valeurs de k

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

resultats = []

for k in [1, 3, 5, 7, 9, 11, 15]:
    modele = KNeighborsClassifier(n_neighbors=k)
    modele.fit(X_train, y_train)
    score_test = modele.score(X_test, y_test)
    resultats.append({"k": k, "score_test": score_test})

pd.DataFrame(resultats)

On teste plusieurs valeurs parce que `k` règle la taille du voisinage.

```text
k petit → voisinage très local, sensible au bruit
k grand → voisinage plus large, plus lissé
```

Il n'existe pas une valeur universelle de `k`. On la choisit en observant les performances.

## 3. Normaliser les données

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

Le scaler apprend la moyenne et l'écart-type sur le train.

Le test est seulement transformé avec les règles apprises sur le train.

C'est important pour ne pas utiliser d'information du test pendant la préparation du modèle.

## 4. Tester plusieurs valeurs de k après normalisation

In [ ]:
resultats_norm = []

for k in [1, 3, 5, 7, 9, 11, 15]:
    modele = KNeighborsClassifier(n_neighbors=k)
    modele.fit(X_train_norm, y_train)
    score_test = modele.score(X_test_norm, y_test)
    resultats_norm.append({"k": k, "score_test_normalise": score_test})

pd.DataFrame(resultats_norm)

Sur Iris, les mesures sont déjà assez comparables, donc la normalisation ne change pas forcément beaucoup le score.

Mais la règle reste importante : KNN dépend des distances, donc il est souvent sensible aux échelles.

## 5. Comparer les voisins avant et après normalisation

In [ ]:
index = X_test.index[0]
fleur = X_test.loc[[index]]
fleur_norm = scaler.transform(fleur)

modele_brut = KNeighborsClassifier(n_neighbors=5)
modele_brut.fit(X_train, y_train)

modele_norm = KNeighborsClassifier(n_neighbors=5)
modele_norm.fit(X_train_norm, y_train)

distances_brut, indices_brut = modele_brut.kneighbors(fleur)
distances_norm, indices_norm = modele_norm.kneighbors(fleur_norm)

print("Indices voisins sans normalisation :", indices_brut[0])
print("Distances sans normalisation       :", distances_brut[0])
print("Indices voisins avec normalisation :", indices_norm[0])
print("Distances avec normalisation       :", distances_norm[0])

Les voisins peuvent changer après normalisation, car la distance n'est plus calculée sur les mêmes échelles.

C'est exactement le point clé :

```text
changer l'échelle des colonnes peut changer la notion de proximité
```

Et comme KNN dépend de la proximité, cela peut changer la prédiction.